In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import inventory as I
import manifest as M
import assemble as A
import concepts as C

smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")

### What is in the assembled tables
Prints, for each site, every assembled table with its row count and column names (no values). RBH tables are mostly wide (one column per measurement) and GOSH has one file per variable, so these names are what the concept map will be written from.

In [ ]:
assembled_dir = open_picu_dir / "derived" / "assembled"
for site in ["RBH", "GOSH"]:
    C.list_sources(assembled_dir, site)

### Cross-site concept map
Lists the long GOSH label columns first (so GOSH blood gases, fluids and drugs can be mapped), then measures every source in the map at all three sites: stays with a value, rows, share numeric and the 5th, median and 95th percentile. The percentiles show unit differences between sites. Counts and ranges only. This reads every assembled table, so it takes a few minutes.

In [ ]:
importlib.reload(C)
C.list_values(assembled_dir, "GOSH", "merged_vent_data__vent_Ventilation_mode_flowsheet_rows",
              "res_value", where=("var_type", "Ventilation"))

In [ ]:
G = "GOSH"
C.list_values(assembled_dir, G, "merged_gas_data__merged_blood_gasses", "standard_variable")
C.list_values(assembled_dir, G, "merged_balance__Net_Bal", "shortLabel")
C.list_values(assembled_dir, G, "merged_balance__Total_fluids", "shortLabel")
C.list_values(assembled_dir, G, "merged_vent_data__vent_Ventilation_mode_flowsheet_rows", "DisplayName")

In [ ]:
st = pd.read_parquet(open_picu_dir / "derived" / "static" / "pooled_static.parquet")
n_stays = st.groupby("site").size().to_dict()
detail, summary = C.build(assembled_dir, n_stays=n_stays,
                          save_to=open_picu_dir / "derived" / "concepts" / "concept_map.csv")
summary.drop(columns="note")

In [ ]:
C.survivors(summary, min_pct=50)[["concept", "group", "RBH_pct", "SMH_pct", "GOSH_pct"]]